# BioMedCLIP fine-tuned on OpenI

Fills the missing cell of the 2x2 comparison:

| | zero-shot | fine-tuned on OpenI |
|---|---|---|
| CLIP + ClinicalBERT | gap 0.0005 | gap 0.1717 |
| BioMedCLIP | gap 0.0154 | **this notebook** |

Three LR arms (`1e-5 / 3e-6 / 1e-6`); everything else is held identical to `configs/clipnorm.yaml`.
The high arm reuses the LR tuned for randomly-aligned towers and is the one most at risk of
catastrophic forgetting -- the sweep exists so a bad result can be attributed to LR, not to
the backbone.

**Requires:** GPU, internet, dataset `raddar/chest-xrays-indiana-university`.

In [ ]:
!pip install -q open_clip_torch>=3.0 transformers
!git clone -q https://github.com/maxzhang646/medical-clip.git
import sys
sys.path.insert(0, 'medical-clip/src')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Locate the OpenI mount and confirm the split files travelled with the repo.
from pathlib import Path

CANDIDATES = list(Path('/kaggle/input').glob('*indiana*')) + list(Path('/kaggle/input').glob('*chest-xrays*'))
INDIANA_DIR = None
for c in CANDIDATES:
    for d in [c] + [p for p in c.iterdir() if p.is_dir()]:
        if (d / 'indiana_reports.csv').exists():
            INDIANA_DIR = d
            break
    if INDIANA_DIR:
        break
assert INDIANA_DIR, f'OpenI not found. Mounted: {[p.name for p in Path("/kaggle/input").iterdir()]}'
print('INDIANA_DIR =', INDIANA_DIR)
print('images_normalized:', (INDIANA_DIR / 'images' / 'images_normalized').exists())

SPLIT_DIR = Path('medical-clip/splits')
for s in ['train', 'val', 'test']:
    f = SPLIT_DIR / f'openi_{s}_uids.txt'
    print(f'{s}: {len(f.read_text().split())} uids')

In [ ]:
# Sanity check before spending GPU hours: pretrained alignment must survive the pipeline.
# Initial InfoNCE should be well below ln(batch_size); at ~ln(N) the wiring is wrong.
!cd medical-clip && python3 scripts/stage4_biomedclip_smoke.py \
    --indiana-dir {INDIANA_DIR} --skip-medclip-check

In [ ]:
import subprocess, time

ARMS = {'lr1e5': '1.0e-5', 'lr3e6': '3.0e-6', 'lr1e6': '1.0e-6'}
logs = {}

for arm in ARMS:
    print(f'\n{"="*70}\n{arm}  (lr_encoders={ARMS[arm]})\n{"="*70}', flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        ['python3', 'src/train.py',
         '--config', f'configs/biomedclip_ft_{arm}.yaml',
         '--indiana-dir', str(INDIANA_DIR)],
        cwd='medical-clip', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    proc.wait()
    logs[arm] = ''.join(lines)
    print(f'--- {arm} finished in {(time.time()-t0)/60:.1f} min, rc={proc.returncode}', flush=True)
    assert proc.returncode == 0, f'{arm} failed'

In [ ]:
# Evaluate every arm on the OpenI test split: R@K / MedR + matched-minus-random gap.
import numpy as np, yaml
from torch.utils.data import DataLoader

from biomedclip_model import BioMedCLIPFinetune, build_tokenize_fn
from dataset import OpenIDataset
from retrieval import recall_at_k

device = torch.device('cuda')
tokenize_fn = build_tokenize_fn()

def evaluate_checkpoint(ckpt_path, freeze_image_layers=8):
    model = BioMedCLIPFinetune(freeze_image_layers=freeze_image_layers)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    model = model.to(device).eval()

    ds = OpenIDataset(str(INDIANA_DIR), tokenizer=None, split='test',
                      split_dir=str(SPLIT_DIR),
                      transform=model.preprocess_val, tokenize_fn=tokenize_fn)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)

    I, T = [], []
    with torch.no_grad():
        for b in loader:
            I.append(model.encode_image(b['image'].to(device)).cpu())
            T.append(model.encode_text(b['input_ids'].to(device)).cpu())
    I, T = torch.cat(I).numpy(), torch.cat(T).numpy()
    sim = I @ T.T

    n = sim.shape[0]
    matched = float(np.diag(sim).mean())
    off = sim[~np.eye(n, dtype=bool)]
    del model; torch.cuda.empty_cache()
    return {'i2t': recall_at_k(sim, [1, 5, 10]), 't2i': recall_at_k(sim.T, [1, 5, 10]),
            'matched': matched, 'random': float(off.mean()), 'gap': matched - float(off.mean())}

results = {}
for arm in ARMS:
    ckpt = f'medical-clip/checkpoints_biomedclip_{arm}/best.pt'
    results[arm] = evaluate_checkpoint(ckpt)
    print(arm, results[arm], flush=True)

In [ ]:
# Final table, with the two published reference rows for context.
REFERENCE = {
    'Vanilla OpenAI CLIP (zero-shot)':      (0.00, 1.56, 3.12, 162.50, 0.31, 2.81, 4.06, 166.00, 0.0005),
    'BioMedCLIP (zero-shot)':               (1.56, 5.31, 8.12, 120.00, 1.25, 6.25, 8.75, 108.50, 0.0154),
    'CLIP+ClinicalBERT (CLIP-norm, FT)':    (3.75, 12.81, 20.62, 49.50, 4.38, 11.88, 19.38, 47.00, 0.1717),
}

header = '| Model | I→T R@1 | R@5 | R@10 | MedR | T→I R@1 | R@5 | R@10 | MedR | Gap |'
rows = [header, '|' + '---|' * 10]
for name, v in REFERENCE.items():
    rows.append('| ' + name + ' | ' + ' | '.join(f'{x:.2f}' for x in v[:-1]) + f' | {v[-1]:.4f} |')
for arm, r in results.items():
    i, t = r['i2t'], r['t2i']
    rows.append(f"| **BioMedCLIP FT ({ARMS[arm]})** | {i['R@1']:.2f} | {i['R@5']:.2f} | {i['R@10']:.2f} | "
                f"{i['MedR']:.2f} | {t['R@1']:.2f} | {t['R@5']:.2f} | {t['R@10']:.2f} | {t['MedR']:.2f} | {r['gap']:.4f} |")
table = '\n'.join(rows)
print(table)

with open('/kaggle/working/stage4_biomedclip_finetune.md', 'w') as f:
    f.write('# Stage 4: BioMedCLIP fine-tuned on OpenI\n\n' + table + '\n\n## Training logs\n\n')
    for arm, log in logs.items():
        f.write(f'### {arm} (lr_encoders={ARMS[arm]})\n\n```\n' + log + '\n```\n\n')
print('\nwrote /kaggle/working/stage4_biomedclip_finetune.md')

In [ ]:
# Keep only the best arm's checkpoint (Kaggle output quota); download it from the Output tab.
import shutil
best_arm = max(results, key=lambda a: results[a]['gap'])
print('best arm by matched-random gap:', best_arm)
shutil.copy(f'medical-clip/checkpoints_biomedclip_{best_arm}/best.pt',
            f'/kaggle/working/biomedclip_ft_{best_arm}_best.pt')
!ls -lh /kaggle/working